# Cinemática inversa de un manipulador planar de 6 articulaciones

Asignatura: Introducción a la Robótica Industrial

El objetivo es implementar la cinemática inversa de un manipulador planar de seis articulaciones mediante el método del Jacobiano pseudoinverso. El end effector del manipulador debe desplazarse hacia una posición objetivo definida en el plano XY.

In [3]:
# Librerías y parámetros

import numpy as np
import math

# Longitud de los seis eslabones
lengths = [100] * 6

# Configuración inicial
thetas = [0, math.pi/2, 0, 0, 0, 0]

print("Longitudes:", lengths)
print("Ángulos iniciales:", thetas)

Longitudes: [100, 100, 100, 100, 100, 100]
Ángulos iniciales: [0, 1.5707963267948966, 0, 0, 0, 0]


## Cinemática directa

Para un manipulador planar, la orientación de cada eslabón depende de la suma de los ángulos de las articulaciones anteriores.

La posición del efector final se obtiene mediante:

\[
x = \sum_{i=1}^{6} L_i \cos\left(\sum_{k=1}^{i}\theta_k\right)
\]

\[
y = \sum_{i=1}^{6} L_i \sin\left(\sum_{k=1}^{i}\theta_k\right)
\]

donde \(L_i\) representa la longitud de cada eslabón y \(\theta_i\) el ángulo de cada articulación.

In [ ]:
# Cinemática directa

def forward_kinematics(thetas, lengths):
    x = 0
    y = 0

    for i in range(len(thetas)):
        angle = sum(thetas[:i+1])

        x += lengths[i] * math.cos(angle)
        y += lengths[i] * math.sin(angle)

    return np.array([x, y])


position = forward_kinematics(thetas, lengths)

print("Posición end effector:", position)

Posición final: [100. 500.]


## Matriz Jacobiana

El Jacobiano relaciona pequeñas variaciones de las articulaciones con variaciones de la posición del end effector:

\[
\Delta p = J\Delta\theta
\]

Para seis articulaciones y una posición definida mediante dos coordenadas, el Jacobiano tiene dimensiones \(2\times6\):

\[
J =
\begin{bmatrix}
\frac{\partial x}{\partial\theta_1} &
\frac{\partial x}{\partial\theta_2} &
\cdots &
\frac{\partial x}{\partial\theta_6}\\
\frac{\partial y}{\partial\theta_1} &
\frac{\partial y}{\partial\theta_2} &
\cdots &
\frac{\partial y}{\partial\theta_6}
\end{bmatrix}
\]

Para cada articulación \(i\):

\[
\frac{\partial x}{\partial\theta_i}
=
-\sum_{j=i}^{6}L_j\sin(\phi_j)
\]

\[
\frac{\partial y}{\partial\theta_i}
=
\sum_{j=i}^{6}L_j\cos(\phi_j)
\]

siendo:

\[
\phi_j=\sum_{k=1}^{j}\theta_k
\]

In [6]:
# Cálculo del jacobiano

def jacobian(thetas, lengths):
    n = len(thetas)

    J = np.zeros((2, n))

    for i in range(n):
        for j in range(i, n):

            angle = sum(thetas[:j+1])

            J[0, i] += -lengths[j] * math.sin(angle)
            J[1, i] +=  lengths[j] * math.cos(angle)

    return J


J = jacobian(thetas, lengths)

print("Jacobiano:")
print(J)

print("\nDimensiones:", J.shape)

Jacobiano:
[[-5.0000000e+02 -5.0000000e+02 -4.0000000e+02 -3.0000000e+02
  -2.0000000e+02 -1.0000000e+02]
 [ 1.0000000e+02  3.0616170e-14  2.4492936e-14  1.8369702e-14
   1.2246468e-14  6.1232340e-15]]

Dimensiones: (2, 6)


## Cinemática inversa mediante pseudoinversa

Se desea determinar la variación de los seis ángulos necesaria para aproximar el end effector a una posición objetivo.

El error de posición se define como:

\[
e =
\begin{bmatrix}
x_d-x\\
y_d-y
\end{bmatrix}
\]

Como el Jacobiano tiene dimensiones \(2\times6\), no es una matriz cuadrada y no posee una inversa convencional.

Por este motivo se utiliza la pseudoinversa de Moore-Penrose:

\[
J^+ = \operatorname{pinv}(J)
\]

La variación de los ángulos se obtiene entonces mediante:

\[
\boxed{\Delta\theta=J^+e}
\]

El vector resultante contiene las variaciones de las seis articulaciones:

\[
\Delta\theta =
\begin{bmatrix}
\Delta\theta_1 &
\Delta\theta_2 &
\Delta\theta_3 &
\Delta\theta_4 &
\Delta\theta_5 &
\Delta\theta_6
\end{bmatrix}^{T}
\]

In [7]:
# Ejemplo

# Posición actual
current = forward_kinematics(thetas, lengths)

# Posición objetivo de ejemplo
target = np.array([200, 300])

# Error de posición
error = target - current

# Jacobiano
J = jacobian(thetas, lengths)

# Pseudoinversa
J_pinv = np.linalg.pinv(J)

# Incremento necesario de los ángulos
delta_theta = J_pinv @ error

print("Posición actual:", current)
print("Objetivo:", target)
print("Error:", error)

print("\nPseudoinversa:")
print(J_pinv)

print("\nDelta theta:")
print(delta_theta)

print("\nDelta theta en grados:")
print(np.degrees(delta_theta))

Posición actual: [100. 500.]
Objetivo: [200 300]
Error: [ 100. -200.]

Pseudoinversa:
[[ 6.66599405e-19  1.00000000e-02]
 [-9.09090909e-04 -4.54545455e-03]
 [-7.27272727e-04 -3.63636364e-03]
 [-5.45454545e-04 -2.72727273e-03]
 [-3.63636364e-04 -1.81818182e-03]
 [-1.81818182e-04 -9.09090909e-04]]

Delta theta:
[-2.          0.81818182  0.65454545  0.49090909  0.32727273  0.16363636]

Delta theta en grados:
[-114.59155903   46.87836506   37.50269204   28.12701903   18.75134602
    9.37567301]


## Implementación en el simulador

El procedimiento anterior se implementó en el método `control6J2D(self, target)` de la clase `JacobianController`.

En cada iteración del simulador se realiza el siguiente procedimiento:

1. Se obtiene la posición actual del end effector.
2. Se calcula el error entre la posición objetivo y la posición actual.
3. Se calcula el Jacobiano del manipulador en su configuración actual.
4. Se obtiene la pseudoinversa mediante `numpy.linalg.pinv()`.
5. Se calcula el vector de incrementos articulares mediante \(\Delta\theta=J^+e\).
6. Se limita el incremento máximo para evitar movimientos excesivamente grandes.
7. Se actualizan las seis articulaciones mediante `Arm.move()`.

Este procedimiento se repite continuamente, permitiendo que el end effector siga la posición del cursor utilizada como objetivo.